# 📰 Fake News Detector — EDA & Training Walkthrough
**NLP Final Year Project**

This notebook walks through the complete machine learning pipeline:
1. Dataset loading & exploration
2. Text preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature engineering (TF-IDF)
5. Model training & comparison
6. Evaluation & interpretation


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn nltk pandas matplotlib seaborn wordcloud lime joblib

In [ ]:
import re, string, warnings, json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

import joblib

STOP_WORDS = set(stopwords.words('english'))
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print("✅ All imports loaded.")

## 1. Load Dataset

In [ ]:
# Load True and Fake news CSVs
true_df = pd.read_csv('dataset/True.csv')
fake_df = pd.read_csv('dataset/Fake.csv')

true_df['label'] = 1   # Real
fake_df['label'] = 0   # Fake

df = pd.concat([true_df, fake_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total articles : {len(df):,}")
print(f"Real (label=1) : {df['label'].sum():,}")
print(f"Fake (label=0) : {(df['label']==0).sum():,}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Class distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['label'].value_counts()
axes[0].bar(['Fake', 'Real'], [counts[0], counts[1]], color=['#e63946','#2dc653'], width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of articles')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Article length distribution
df['text_len'] = df['text'].fillna('').apply(len)
axes[1].hist(df[df['label']==1]['text_len'], bins=50, alpha=0.7, color='#2dc653', label='Real', density=True)
axes[1].hist(df[df['label']==0]['text_len'], bins=50, alpha=0.7, color='#e63946', label='Fake', density=True)
axes[1].set_title('Article Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Character count')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].set_xlim(0, 30000)

plt.tight_layout()
plt.show()
print(f"Avg real length: {df[df['label']==1]['text_len'].mean():.0f} chars")
print(f"Avg fake length: {df[df['label']==0]['text_len'].mean():.0f} chars")

In [ ]:
# ── Subject/category breakdown ──
if 'subject' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, label, color, title in zip(axes, [1, 0], ['#2dc653','#e63946'], ['Real News — Subjects','Fake News — Subjects']):
        sub = df[df['label']==label]['subject'].value_counts().head(8)
        ax.barh(sub.index[::-1], sub.values[::-1], color=color, alpha=0.85)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Word clouds ──
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, label, cmap, title in zip(axes, [1,0], ['Greens','Reds'], ['Real News','Fake News']):
        text = ' '.join(df[df['label']==label]['text'].fillna('').tolist())
        wc = WordCloud(width=600, height=300, background_color='white',
                       colormap=cmap, max_words=100, stopwords=STOP_WORDS).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f'Word Cloud — {title}', fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install wordcloud: pip install wordcloud")

## 3. Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """
    Lowercase → remove URLs → remove punctuation → remove digits → remove stopwords.
    This exact function is used in both train_model.py and app.py to avoid training-serving skew.
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 1]
    return ' '.join(tokens)

# Test it
sample = "BREAKING: The President signed a NEW executive order today! Visit http://example.com for details."
print("Before:", sample)
print("After: ", clean_text(sample))

In [ ]:
# Apply to full dataset
df['cleaned'] = df['text'].fillna('').apply(clean_text)
df['cleaned_len'] = df['cleaned'].apply(len)

print(f"Average cleaned length — Real: {df[df['label']==1]['cleaned_len'].mean():.0f}  |  Fake: {df[df['label']==0]['cleaned_len'].mean():.0f}")
df[['text','cleaned','label']].head(3)

## 4. Feature Engineering — TF-IDF

In [ ]:
X = df['cleaned']
y = df['label']

# Stratified split (80/20) — vectorizer must only see training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True     # log scaling to dampen high-frequency terms
)
X_train_vec = vectorizer.fit_transform(X_train)   # fit ONLY on training data
X_test_vec  = vectorizer.transform(X_test)         # transform test without re-fitting

print(f"Train matrix : {X_train_vec.shape}")
print(f"Test matrix  : {X_test_vec.shape}")
print(f"Vocabulary   : {len(vectorizer.vocabulary_):,} features")

In [ ]:
# ── Top TF-IDF terms per class ──
# Average TF-IDF score across each class
real_idx = y_train[y_train==1].index.intersection(pd.Index(range(X_train_vec.shape[0])))
feature_names = vectorizer.get_feature_names_out()

# Get mean TF-IDF for each class
train_labels = y_train.reset_index(drop=True)
X_dense = X_train_vec

real_mask = train_labels == 1
fake_mask = train_labels == 0

real_means = np.asarray(X_dense[real_mask].mean(axis=0)).flatten()
fake_means = np.asarray(X_dense[fake_mask].mean(axis=0)).flatten()

top_real = pd.Series(real_means, index=feature_names).nlargest(15)
top_fake = pd.Series(fake_means, index=feature_names).nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_real[::-1].plot.barh(ax=axes[0], color='#2dc653', alpha=0.85)
axes[0].set_title('Top TF-IDF terms — Real News', fontsize=12, fontweight='bold')
top_fake[::-1].plot.barh(ax=axes[1], color='#e63946', alpha=0.85)
axes[1].set_title('Top TF-IDF terms — Fake News', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Model Training & Comparison

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, solver='lbfgs', random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Linear SVM':          CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)),
}

results = {}
for name, clf in MODELS.items():
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1 Score':  f1_score(y_test, y_pred, zero_division=0),
        'clf':       clf,
        'y_pred':    y_pred,
    }
    print(f"  {name:<25}  Acc: {results[name]['Accuracy']*100:.2f}%  F1: {results[name]['F1 Score']*100:.2f}%")

print("\n✅ All models trained.")

In [ ]:
# ── Comparison bar chart ──
metrics_names = ['Accuracy','Precision','Recall','F1 Score']
x = np.arange(len(metrics_names))
width = 0.25
colors = ['#4361ee','#e63946','#2dc653']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics_names]
    ax.bar(x + i*width, vals, width, label=name, color=colors[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({k: {m: f"{v*100:.2f}%" for m,v in res.items() if m in metrics_names}
                        for k, res in results.items()}).T
print(summary)

## 6. Evaluation — Best Model

In [ ]:
# Pick best model by F1
best_name = max(results, key=lambda k: results[k]['F1 Score'])
best = results[best_name]
print(f"Best model: {best_name}  (F1 = {best['F1 Score']*100:.2f}%)")

# Full classification report
print("\n" + classification_report(y_test, best['y_pred'], target_names=['Fake','Real']))

In [ ]:
# ── Confusion matrix ──
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best['y_pred'],
    display_labels=['Fake','Real'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. LIME — Word-Level Explanation

In [ ]:
try:
    from lime.lime_text import LimeTextExplainer

    best_clf = best['clf']
    explainer = LimeTextExplainer(class_names=['Fake','Real'])

    def predict_fn(texts):
        vecs = vectorizer.transform([clean_text(t) for t in texts])
        return best_clf.predict_proba(vecs)

    # Pick a test article
    sample_text = X_test.iloc[0]
    true_label  = y_test.iloc[0]
    pred_label  = best_clf.predict(vectorizer.transform([clean_text(sample_text)]))[0]

    print(f"True label : {'Real' if true_label==1 else 'Fake'}")
    print(f"Predicted  : {'Real' if pred_label==1 else 'Fake'}")
    print(f"\nText preview:\n{sample_text[:300]}...")

    exp = explainer.explain_instance(sample_text, predict_fn, num_features=10, num_samples=300)
    exp.show_in_notebook(text=True)

except ImportError:
    print("Install lime: pip install lime")

## 8. Save Artefacts

In [ ]:
joblib.dump(best['clf'], 'model.pkl')
joblib.dump(vectorizer,  'vectorizer.pkl')

metrics_out = {
    'best_model':       best_name,
    'accuracy':         round(best['Accuracy'],  4),
    'precision':        round(best['Precision'], 4),
    'recall':           round(best['Recall'],    4),
    'f1_score':         round(best['F1 Score'],  4),
    'confusion_matrix': confusion_matrix(y_test, best['y_pred']).tolist(),
    'vocab_size':       len(vectorizer.vocabulary_),
    'ngram_range':      '1-2',
    'max_features':     10_000,
    'all_models': {
        name: {
            'accuracy':  round(res['Accuracy'],  4),
            'precision': round(res['Precision'], 4),
            'recall':    round(res['Recall'],    4),
            'f1_score':  round(res['F1 Score'],  4),
            'confusion_matrix': confusion_matrix(y_test, res['y_pred']).tolist(),
        } for name, res in results.items()
    }
}

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print("✅ Saved: model.pkl  |  vectorizer.pkl  |  metrics.json")
print(f"   Best model : {best_name}")
print(f"   Accuracy   : {best['Accuracy']*100:.2f}%")
print(f"   F1 Score   : {best['F1 Score']*100:.2f}%")
print("\nRun the app:  streamlit run app.py")